# 05 — Efisiensi: Latensi & Puncak VRAM (RQ4)

**Tujuan:** membuktikan bahwa sistem RAG **lokal** (on-premise) bukan hanya akurat dan aman, tetapi juga **terukur biayanya**. Inilah yang membedakannya dari sekadar "membangun aplikasi".

| Bagian | Metrik | Satuan |
|---|---|---|
| Retrieval | latensi encode + pencarian vektor | milidetik (ms) |
| Generasi | TTFT (*time to first token*), total latensi | detik (s) |
| Throughput | token keluaran per detik | token/s |
| Sumber daya | puncak VRAM GPU & RAM proses | MB |

> **Prasyarat:** notebook 02 sudah membuat indeks, dan server Ollama berjalan.
> Jika mesin tidak punya GPU NVIDIA, `nvidia-smi` gagal dengan sopan → notebook tetap mencatat **RAM** dan menandai `gpu_available=false`. Catat itu sebagai batasan (*limitation*) di laporan.

In [ ]:
import json
import re
import statistics
import subprocess
import threading
import time
from datetime import date
from pathlib import Path

import chromadb
import ollama
import pandas as pd
import psutil
from sentence_transformers import SentenceTransformer

# --- parameter (samakan dengan notebook 02) ---
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64
EMBED_MODEL = "BAAI/bge-m3"
LLM_MODEL = "llama3.1:8b-instruct-q4_K_M"
K = 5

TAG = f"cs{CHUNK_SIZE}_ov{CHUNK_OVERLAP}_{EMBED_MODEL.split('/')[-1]}"
COLLECTION = re.sub(r"[^a-zA-Z0-9_]", "_", f"rag_{TAG}")

VECTOR_DIR = Path("vectorstore")
RESULT_DIR = Path("eval/results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Tag        :", TAG)
print("Collection :", COLLECTION)
print("LLM        :", LLM_MODEL)
print("CPU count  :", psutil.cpu_count(logical=True))

## 0. Monitor sumber daya (VRAM + RAM)

`nvidia-smi` dibaca dari proses terpisah (thread), karena VRAM dipakai oleh server **Ollama**, bukan proses Python ini.

In [ ]:
class ResourceMonitor:
    "Mencatat puncak VRAM GPU (lewat nvidia-smi) dan RAM (psutil) selama benchmark."

    def __init__(self, interval=0.2):
        self.interval = interval
        self._running = False
        self._thread = None
        self.peak_vram_mb = 0.0
        self.peak_ram_mb = 0.0
        self.proc = psutil.Process()

    @staticmethod
    def _gpu_vram_mb():
        try:
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                stderr=subprocess.DEVNULL, text=True, timeout=5,
            )
            vals = [float(x) for x in out.strip().splitlines() if x.strip()]
            return max(vals) if vals else None
        except Exception:
            return None

    def _loop(self):
        while self._running:
            vram = self._gpu_vram_mb()
            if vram is not None:
                self.peak_vram_mb = max(self.peak_vram_mb, vram)
            try:
                ram = self.proc.memory_info().rss / 1024 / 1024
                for child in self.proc.children(recursive=True):
                    try:
                        ram += child.memory_info().rss / 1024 / 1024
                    except Exception:
                        pass
                self.peak_ram_mb = max(self.peak_ram_mb, ram)
            except Exception:
                pass
            time.sleep(self.interval)

    def start(self):
        self._running = True
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        if self._thread:
            self._thread.join(timeout=2)

    def reset(self):
        self.peak_vram_mb = 0.0
        self.peak_ram_mb = 0.0


monitor = ResourceMonitor()
print("Monitor siap. GPU terdeteksi:", monitor._gpu_vram_mb() is not None)

## 1. Muat indeks & model embedding

In [ ]:
model = SentenceTransformer(EMBED_MODEL)
client = chromadb.PersistentClient(path=str(VECTOR_DIR))
collection = client.get_collection(COLLECTION)
print("Collection:", COLLECTION, "| jumlah vektor:", collection.count())

## 2. Muat daftar pertanyaan uji

Efisiensi diukur pada **himpunan pertanyaan yang sama** dengan notebook 03, agar hasil dapat dibandingkan.

In [ ]:
gold = pd.read_csv("eval/gold_qa.csv")
gold = gold[~gold["sumber"].astype(str).str.contains("<ganti", na=False)].reset_index(drop=True)
questions = gold["pertanyaan"].tolist()
print("Jumlah pertanyaan:", len(questions))

## 3. Latensi retrieval

Waktu untuk mengubah pertanyaan menjadi vektor + mengambil top-k chunk terdekat.

In [ ]:
monitor.reset()
monitor.start()
retrieval_times = []
for q in questions:
    t0 = time.perf_counter()
    q_emb = model.encode([q], normalize_embeddings=True).tolist()
    collection.query(query_embeddings=q_emb, n_results=K)
    retrieval_times.append((time.perf_counter() - t0) * 1000)
monitor.stop()

retrieval_retr = {
    "mean_ms": round(statistics.mean(retrieval_times), 1),
    "median_ms": round(statistics.median(retrieval_times), 1),
}
print("Retrieval (ms): mean=%.1f  median=%.1f" % (retrieval_retr["mean_ms"], retrieval_retr["median_ms"]))

## 4. Latensi generasi (streaming: TTFT + total + token/s)

Mode *stream* dipakai agar **TTFT** (waktu sampai token pertama) terukur — inilah angka yang dirasakan pengguna saat mengetik pertanyaan.

In [ ]:
def stream_answer(question, k=K):
    q_emb = model.encode([question], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=k)
    metas, docs = res["metadatas"][0], res["documents"][0]
    context = "\n\n".join(f"[{m['source']}] {d}" for m, d in zip(metas, docs))
    prompt = "Jawab HANYA berdasarkan konteks.\n\nKonteks:\n" + context + f"\n\nPertanyaan: {question}\nJawaban:"

    t0 = time.perf_counter()
    ttft = None
    parts = []
    final = None
    for chunk in ollama.chat(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}], stream=True):
        if ttft is None:
            ttft = time.perf_counter() - t0
        parts.append(chunk["message"]["content"])
        if chunk.get("done"):
            final = chunk
    total = time.perf_counter() - t0

    eval_count = (final or {}).get("eval_count")
    eval_ns = (final or {}).get("eval_duration")
    tok_per_s = (eval_count / (eval_ns / 1e9)) if (eval_count and eval_ns) else None
    return {"ttft_s": ttft, "total_s": total, "tokens": eval_count, "tok_per_s": tok_per_s}

## 5. Jalankan benchmark penuh

Semua pertanyaan dijalankan sekali; VRAM & RAM dipantau sepanjang proses ini (puncak generasi).

In [ ]:
rows = []
monitor.reset()
monitor.start()
for q in questions:
    r = stream_answer(q)
    rows.append({"question": q[:80], **r})
monitor.stop()

eff = pd.DataFrame(rows)
eff.describe(percentiles=[0.5, 0.95])

## 6. Ringkasan + simpan

Semua angka disimpan ke `eval/results/` agar setiap klaim di BAB IV dapat dilacak.

In [ ]:
def p95(values):
    vals = sorted(v for v in values if v is not None)
    if not vals:
        return None
    idx = min(len(vals) - 1, int(round(0.95 * (len(vals) - 1))))
    return vals[idx]


tok = eff["tok_per_s"].dropna().tolist()
summary = {
    "date": date.today().isoformat(),
    "tag": TAG,
    "llm": LLM_MODEL,
    "embed_model": EMBED_MODEL,
    "k": K,
    "n_questions": int(len(eff)),
    "retrieval_ms": {
        "mean": round(statistics.mean(retrieval_times), 1),
        "median": round(statistics.median(retrieval_times), 1),
        "p95": round(p95(retrieval_times), 1),
    },
    "generation": {
        "ttft_s_mean": round(eff["ttft_s"].mean(), 3),
        "total_s_mean": round(eff["total_s"].mean(), 3),
        "total_s_median": round(eff["total_s"].median(), 3),
        "total_s_p95": round(p95(eff["total_s"].tolist()), 3),
        "tok_per_s_mean": round(statistics.mean(tok), 2) if tok else None,
    },
    "peak_vram_mb": round(monitor.peak_vram_mb) or None,
    "peak_ram_mb": round(monitor.peak_ram_mb) or None,
    "gpu_available": monitor.peak_vram_mb > 0,
}

(RESULT_DIR / f"efficiency_{TAG}.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
eff.to_csv(RESULT_DIR / f"efficiency_{TAG}.csv", index=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))

## 7. Cara menafsirkan & langkah berikutnya

**Menafsirkan:**
- **TTFT** kecil (< 1–2 s) = terasa responsif.
- **token/s** kecil pada CPU = wajar; catat sebagai batasan bila tanpa GPU.
- **Puncak VRAM** menentukan apakah skripsi ini bisa dijalankan pada laptop lab / server perusahaan tertentu.

**Langkah berikutnya:**
1. Jalankan notebook ini untuk **tiap konfigurasi** yang dibandingkan di RQ2 (chunk/embedding/LLM lain).
2. Susun tabel gabungan: konfigurasi → akurasi (nb 03) → keamanan (nb 04) → efisiensi (nb 05).
3. Grafik untuk BAB IV: scatter **akurasi vs latensi** dan **akurasi vs VRAM** (trade-off).
4. Cantumkan spesifikasi mesin uji (CPU, RAM, GPU, VRAM, versi Ollama) di bagian metodologi.

> **Konsistensi:** `TAG` harus sama dengan notebook 02/03 agar `collection` yang diukur identik.